In [1]:
# Import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
# Load the datasets
df1 = pd.read_csv('spam.csv', encoding='latin-1')
df2 = pd.read_csv('SMS Spam Dataset.csv', encoding='latin-1')

# Preview both datasets
print("Dataset 1 — spam.csv:")
print(df1.head())
print("\nShape:", df1.shape)
print("\n---")
print("Dataset 2 — SMS Spam Dataset.csv:")
print(df2.head())
print("\nShape:", df2.shape)

Dataset 1 — spam.csv:
     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  

Shape: (5572, 5)

---
Dataset 2 — SMS Spam Dataset.csv:
                                                 sms  label
0  Go until jurong point, crazy.. Available only ...      0
1                    Ok lar... Joking wif u oni...\n      0
2  Free entry in 2 a wkly comp to win FA Cup fina...      1
3  U dun say so early hor... U c already then say...      0
4  Nah I don't think he go

In [3]:
# Clean Dataset 1
df1 = df1[['v1', 'v2']]
df1.columns = ['label', 'message']
df1['label'] = df1['label'].map({'ham': 0, 'spam': 1})

# Clean Dataset 2
df2.columns = ['message', 'label']
df2 = df2[['label', 'message']]

# Combine both datasets
df = pd.concat([df1, df2], ignore_index=True)

# Remove duplicates
df = df.drop_duplicates()

# Remove any empty rows
df = df.dropna()

# Check the result
print("Combined Dataset:")
print(df.head(10))
print("\nShape:", df.shape)
print("\nLabel distribution:")
print(df['label'].value_counts())

Combined Dataset:
   label                                            message
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...
5      1  FreeMsg Hey there darling it's been 3 week's n...
6      0  Even my brother is not like to speak with me. ...
7      0  As per your request 'Melle Melle (Oru Minnamin...
8      1  WINNER!! As a valued network customer you have...
9      1  Had your mobile 11 months or more? U R entitle...

Shape: (10340, 2)

Label distribution:
0    9034
1    1306
Name: label, dtype: int64


In [4]:
# Add scam-specific features
import re

def count_urgency_words(text):
    urgency_words = ['urgent', 'immediately', 'now', 'hurry', 'quick', 
                     'fast', 'limited', 'expire', 'deadline', 'asap',
                     'don\'t wait', 'act now', 'before it\'s too late']
    text = text.lower()
    return sum(1 for word in urgency_words if word in text)

def count_pressure_phrases(text):
    pressure_phrases = ['trust me', 'don\'t worry', 'i also did it',
                        'just ignore', 'it\'s fine', 'believe me',
                        'i promise', 'guaranteed', 'no risk']
    text = text.lower()
    return sum(1 for phrase in pressure_phrases if phrase in text)

def count_money_requests(text):
    money_words = ['send money', 'transfer', 'bank account', 'payment',
                   'crypto', 'bitcoin', 'gift card', 'wire', 'cash',
                   'invest', 'profit', 'earn', 'winning', 'prize']
    text = text.lower()
    return sum(1 for word in money_words if word in text)

def count_suspicious_links(text):
    url_pattern = re.compile(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+])+')
    urls = url_pattern.findall(text)
    return len(urls)

def count_secrecy_phrases(text):
    secrecy_words = ['don\'t tell', 'keep secret', 'between us',
                     'don\'t share', 'confidential', 'private',
                     'don\'t show', 'just you and me']
    text = text.lower()
    return sum(1 for phrase in secrecy_words if phrase in text)

# Apply all features to the dataset
df['urgency_count'] = df['message'].apply(count_urgency_words)
df['pressure_count'] = df['message'].apply(count_pressure_phrases)
df['money_request_count'] = df['message'].apply(count_money_requests)
df['suspicious_links'] = df['message'].apply(count_suspicious_links)
df['secrecy_count'] = df['message'].apply(count_secrecy_phrases)
df['message_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(x.split()))

print("Features added successfully!")
print(df.head())
print("\nFeature summary for scam messages:")
print(df[df['label']==1][['urgency_count', 'pressure_count', 
                            'money_request_count', 'suspicious_links',
                            'secrecy_count']].describe())

Features added successfully!
   label                                            message  urgency_count  \
0      0  Go until jurong point, crazy.. Available only ...              0   
1      0                      Ok lar... Joking wif u oni...              0   
2      1  Free entry in 2 a wkly comp to win FA Cup fina...              0   
3      0  U dun say so early hor... U c already then say...              0   
4      0  Nah I don't think he goes to usf, he lives aro...              0   

   pressure_count  money_request_count  suspicious_links  secrecy_count  \
0               0                    0                 0              0   
1               0                    0                 0              0   
2               0                    0                 0              0   
3               0                    0                 0              0   
4               0                    0                 0              0   

   message_length  word_count  
0             111  